# Push Instruction Templates to HuggingFace

This notebook pushes raw prompt templates (without merging with data) to the `star-templates` HuggingFace dataset.

**Subsets:**
- `all` — Full catalog of all approved templates
- `experimental` — 60 curated prompts used in our experiments, split into `tuning` and `evaluation`

## 1. Fetch and Filter Prompts

In [14]:
import os
import requests
import string
import datasets
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

PROMPTLAB_API_URL = 'https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj'
MAX_RETRIES = 10

prompts = None
for i in tqdm(range(MAX_RETRIES), desc="Fetching prompts"):
    api_response = requests.get(url=PROMPTLAB_API_URL)
    if api_response.ok:
        prompts = api_response.json()
        break

if not prompts:
    raise Exception('Failed to fetch prompts from PromptLab API')

print(f"Total prompts fetched: {len(prompts)}")

Fetching prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Total prompts fetched: 365


In [2]:
approved_prompts = [p for p in prompts if p['status'] == 'APPROVED']
print(f"Approved prompts: {len(approved_prompts)}")

Approved prompts: 355


## 2. Merge Prompter Aliases, Anonymize, and Transform Fields

Some prompters used multiple accounts — we merge them before anonymizing.

In [3]:
# Prompter aliases: merge accounts belonging to the same person
PROMPTER_ALIASES = {
    "zaid": ["zaid", "zaid1"],
    "ahmed": ["ahmed", "ahmed6"],
    "irfan": ["irfan", "irfan9"],
}
prompter_alias_map = {v: canonical for canonical, variants in PROMPTER_ALIASES.items() for v in variants}

# Apply prompter merging, then build anonymization mapping
for p in approved_prompts:
    p['_merged_prompter'] = prompter_alias_map.get(p['created_by'], p['created_by'])

unique_prompters = sorted(set(p['_merged_prompter'] for p in approved_prompts))
prompter_map = {name: string.ascii_lowercase[i] for i, name in enumerate(unique_prompters)}

print(f"Unique prompters (after merging): {len(unique_prompters)}")
print(f"Prompter mapping: {prompter_map}")

Unique prompters (after merging): 7
Prompter mapping: {'Dania_Refai': 'a', 'ahmed': 'b', 'irfan': 'c', 'mais': 'd', 'majed.alshaibani': 'e', 'nawaf': 'f', 'zaid': 'g'}


In [4]:
def clean_dataset_subset(subset):
    """Fix known dirty dataset_subset values from the API."""
    if subset.startswith('subtask5.arabic'):
        return 'subtask5.arabic'
    return subset

def transform_prompt(prompt):
    """Transform a raw API prompt into the star-templates schema."""
    return {
        'template_id': prompt['id'],
        'template_name': prompt['name'],
        'template': prompt['template'],
        'tasks': list(prompt['task'].values()),
        'dataset_name': prompt['dataset_name'],
        'dataset_subset': clean_dataset_subset(prompt['dataset_subset']),
        'answer_choices': prompt.get('answer_choices', []),
        'text_direction': prompt['text_direction'],
        'tags': [tag for tag in prompt.get('tags', []) if tag.strip()],
        'prompter_id': prompter_map[prompt['_merged_prompter']],
    }

transformed = [transform_prompt(p) for p in approved_prompts]
print(f"Transformed {len(transformed)} prompts")
transformed[0]

Transformed 355 prompts


{'template_id': 14898,
 'template_name': 'Prompt with zero-shot chain of thoughts',
 'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
 'tasks': ['claim verification'],
 'dataset_name': 'arbml/ANS_stance',
 'dataset_subset': 'default',
 'answer_choices': ['disagree', 'agree', 'other'],
 'text_direction': 'ltr',
 'tags': ['Zero-shot COT'],
 'prompter_id': 'b'}

## 3. Push `all` Subset

In [5]:
all_dataset = datasets.Dataset.from_list(transformed)
print(all_dataset)
all_dataset[0]

Dataset({
    features: ['template_id', 'template_name', 'template', 'tasks', 'dataset_name', 'dataset_subset', 'answer_choices', 'text_direction', 'tags', 'prompter_id'],
    num_rows: 355
})


{'template_id': 14898,
 'template_name': 'Prompt with zero-shot chain of thoughts',
 'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
 'tasks': ['claim verification'],
 'dataset_name': 'arbml/ANS_stance',
 'dataset_subset': 'default',
 'answer_choices': ['disagree', 'agree', 'other'],
 'text_direction': 'ltr',
 'tags': ['Zero-shot COT'],
 'prompter_id': 'b'}

In [6]:
HF_REPO = 'KFUPM-JRCAI/star-dataset-templates'
# Set to True to keep the dataset private on the Hub.
PRIVATE = False

all_dataset.push_to_hub(HF_REPO, config_name='all', token=HF_TOKEN, private=PRIVATE)
print(f"Pushed 'all' subset with {len(all_dataset)} templates")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed 'all' subset with 355 templates


## 4. Build and Push `experimental` Subset

The `experimental` subset contains the 60 prompts specifically selected for our experiments:
- **`tuning` split**: 30 prompts from primary datasets, used for fine-tuning and intra-dataset evaluation
- **`evaluation` split**: 30 prompts from secondary datasets, used for intra-task (cross-dataset) evaluation

In [7]:
# Experiment metadata: maps task -> (primary_dataset, primary_ids, secondary_dataset, secondary_ids)
EXPERIMENT_CONFIG = {
    'dialect_identification': {
        'primary_dataset': 'arbml/AraBench_dev',
        'primary_ids': [14852, 14850, 14789, 14781, 14561],
        'secondary_dataset': 'arbml/Arabic_Dialects_Dataset',
        'secondary_ids': [14102, 14783, 14784, 14790, 14851],
    },
    'machine_translation': {
        'primary_dataset': 'Helsinki-NLP/opus-100',
        'primary_ids': [14684, 14688, 14680, 14682, 14640],
        'secondary_dataset': 'Helsinki-NLP/tatoeba_mt',
        'secondary_ids': [14866, 14867, 14868, 14889, 14890],
    },
    'NLI': {
        'primary_dataset': 'arbml/ArEntail',
        'primary_ids': [14581, 14816, 14818, 14819, 14820],
        'secondary_dataset': 'arbml/ArabicTE',
        'secondary_ids': [14582, 14673, 14724, 14805, 14855],
    },
    'NLU': {
        'primary_dataset': 'arbml/ArabicMMLU',
        'primary_ids': [14571, 14869, 14787, 14797, 14798],
        'secondary_dataset': 'facebook/belebele',
        'secondary_ids': [14854, 14853, 14801, 14800, 14575],
    },
    'sarcasm_detection': {
        'primary_dataset': 'arbml/ArSarcasm_v2',
        'primary_ids': [14779, 14802, 14835, 14837, 14838],
        'secondary_dataset': 'arbml/iSarcasmEval_task_A',
        'secondary_ids': [14602, 14780, 14859, 14860, 14864],
    },
    'summarization': {
        'primary_dataset': 'GEM/xlsum',
        'primary_ids': [14871, 14803, 14856, 14858, 14668],
        'secondary_dataset': 'arbml/AraSum',
        'secondary_ids': [14891, 14857, 14736, 14735, 14628],
    },
}

In [8]:
# Build lookup from template_id to transformed prompt
templates_by_id = {t['template_id']: t for t in transformed}

tuning_records = []
evaluation_records = []

for task_name, config in EXPERIMENT_CONFIG.items():
    # Tuning split: primary dataset prompts
    for pid in config['primary_ids']:
        template = templates_by_id[pid].copy()
        template['experiment_task'] = task_name
        template['dataset_role'] = 'primary'
        template['paired_dataset'] = config['secondary_dataset']
        tuning_records.append(template)

    # Evaluation split: secondary dataset prompts
    for sid in config['secondary_ids']:
        template = templates_by_id[sid].copy()
        template['experiment_task'] = task_name
        template['dataset_role'] = 'secondary'
        template['paired_dataset'] = config['primary_dataset']
        evaluation_records.append(template)

print(f"Tuning split: {len(tuning_records)} prompts")
print(f"Evaluation split: {len(evaluation_records)} prompts")

Tuning split: 30 prompts
Evaluation split: 30 prompts


In [9]:
tuning_dataset = datasets.Dataset.from_list(tuning_records)
evaluation_dataset = datasets.Dataset.from_list(evaluation_records)

experimental_dataset = datasets.DatasetDict({
    'tuning': tuning_dataset,
    'evaluation': evaluation_dataset,
})

print(experimental_dataset)

DatasetDict({
    tuning: Dataset({
        features: ['template_id', 'template_name', 'template', 'tasks', 'dataset_name', 'dataset_subset', 'answer_choices', 'text_direction', 'tags', 'prompter_id', 'experiment_task', 'dataset_role', 'paired_dataset'],
        num_rows: 30
    })
    evaluation: Dataset({
        features: ['template_id', 'template_name', 'template', 'tasks', 'dataset_name', 'dataset_subset', 'answer_choices', 'text_direction', 'tags', 'prompter_id', 'experiment_task', 'dataset_role', 'paired_dataset'],
        num_rows: 30
    })
})


In [10]:
# Preview a tuning record
print("=== Tuning split example ===")
tuning_dataset[0]

=== Tuning split example ===


{'template_id': 14852,
 'template_name': 'Is it from East Arabic or West Arabic?',
 'template': 'Consider this text:\r\n{{arabic}}\xa0\r\nIf it is written as Modern-Standard-Arabic (MSA), then answer with MSA. If it is not an MSA text, then consider the following:\r\n- If you think it is from Eastern Arabic countries, then answer with what you think is the closest, either Qatari or Lebanese.\r\n- If you think it is from Western Arabic countries, then answer with what you think is the closest, either Tunisian, Morrocan, or Egyptian.\r\nThe answer is:\r\n|||\r\n{{answer_choices[label]}}',
 'tasks': ['dialect identification'],
 'dataset_name': 'arbml/AraBench_dev',
 'dataset_subset': 'default',
 'answer_choices': ['Tunisian',
  'MSA',
  'Morrocan',
  'Qatari',
  'Egyptian',
  'Lebanese'],
 'text_direction': 'ltr',
 'tags': [],
 'prompter_id': 'e',
 'experiment_task': 'dialect_identification',
 'dataset_role': 'primary',
 'paired_dataset': 'arbml/Arabic_Dialects_Dataset'}

In [11]:
# Preview an evaluation record
print("=== Evaluation split example ===")
evaluation_dataset[0]

=== Evaluation split example ===


{'template_id': 14102,
 'template_name': 'Identify Dialect',
 'template': 'For this text: {{Text}}, I am not sure what dialect it is. Can you identify the dialect of the previous text given the following options: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %}, {% endif %}{% endfor %}\r\n|||\r\n{{answer_choices[label]}}',
 'tasks': ['dialect identification'],
 'dataset_name': 'arbml/Arabic_Dialects_Dataset',
 'dataset_subset': 'default',
 'answer_choices': ['Levant', 'North Africa', 'Egypt', 'GULF', 'MSA'],
 'text_direction': 'ltr',
 'tags': [],
 'prompter_id': 'b',
 'experiment_task': 'dialect_identification',
 'dataset_role': 'secondary',
 'paired_dataset': 'arbml/AraBench_dev'}

In [12]:
experimental_dataset.push_to_hub(HF_REPO, config_name='experimental', token=HF_TOKEN, private=PRIVATE)
# push_to_hub only applies `private` when creating the repo, so sync it explicitly.
from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).update_repo_settings(repo_id=HF_REPO, repo_type='dataset', private=PRIVATE)

print(f"Pushed 'experimental' subset with tuning ({len(tuning_dataset)}) and evaluation ({len(evaluation_dataset)}) splits")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/642 [00:00<?, ?B/s]

Pushed 'experimental' subset with tuning (30) and evaluation (30) splits


## 5. Upload README

In [18]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj='star_dataset/push_to_hf/star_templates_README.md',
    path_in_repo='README.md',
    repo_id=HF_REPO,
    repo_type='dataset',
)
print(f"Uploaded README to {HF_REPO}")

/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/.venv/lib/python3.12/site-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- The task_categories "text2text-generation" is not in the official list: text-classification, token-classification, table-question-answering, question-answering, zero-shot-classification, translation, summarization, feature-extraction, text-generation, fill-mask, sentence-similarity, text-to-speech, text-to-audio, automatic-speech-recognition, audio-to-audio, audio-classification, audio-text-to-text, voice-activity-detection, depth-estimation, image-classification, object-detection, image-segmentation, text-to-image, image-to-text, image-to-image, image-to-video, unconditional-image-generation, video-classification, reinforcement-learning, robotics, tabular-classification, tabular-regression, tabular-to-text, table-to-text, multiple-choice, text-ranking, text-retrieval, time-series-forecasting, t

Uploaded README to KFUPM-JRCAI/star-dataset-templates


## Summary

Pushed to `star-templates`:
- **`all`** subset: Full catalog of approved templates
- **`experimental`** subset:
  - `tuning` split: 30 prompts from primary datasets (fine-tuning + intra-dataset evaluation)
  - `evaluation` split: 30 prompts from secondary datasets (intra-task evaluation)